# Tesis v5 · B3: CV dentro de cada seed (seeds 8 a 10)
Entorno normal (sin GPU) y **Run all**, en paralelo con los otros dos notebooks.

Reusa de `tesis_backup/v4` los datos sintéticos y la seed 1; guarda en `tesis_backup/v5`. Este notebook solo entrena sus seeds; la consolidación y las figuras las genera B1.

In [ ]:
!pip install -q aif360 catboost lightgbm xgboost shap adjustText


In [ ]:
# ===== CELDA 0 · SETUP (correr primero, en cualquiera de los dos notebooks) =====
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
warnings.filterwarnings('ignore')

# --- carpeta de salida en Drive (v5 = CV dentro de cada seed) ---
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/tesis_backup/v5'
V4 = '/content/drive/MyDrive/tesis_backup/v4'   # de acá se reusan los datos sintéticos
os.makedirs(OUT, exist_ok=True)

SEEDS = list(range(1, 11))
target_col = 'default.payment.next.month'
drop_cols = ['ID', 'SEX', target_col, 'AGE_GROUP']

# --- carga y limpieza (idéntico a Bozza tesis) ---
url = "https://raw.githubusercontent.com/YuChenAmberLu/Data-Science--Credit-Card-Default/master/UCI_Credit_Card.csv"
df = pd.read_csv(url)
df_clean = df.copy()
df_clean['EDUCATION'] = df_clean['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df_clean['MARRIAGE'] = df_clean['MARRIAGE'].replace({0: 3})
pay_cols = [c for c in df_clean.columns if c.startswith('PAY_') and 'AMT' not in c]
for c in pay_cols:
    df_clean[c] = df_clean[c].replace({-2: -1, 0: -1})
df_clean['AGE_GROUP'] = df_clean['AGE'].apply(lambda x: 'core' if 25 <= x <= 65 else 'edge')

# --- inyección de sesgo (idéntico a Bozza tesis: una sola vez, seed 42, antes del split) ---
def inject_bias(df, sex_col='SEX', target_col='default.payment.next.month',
                underpriv_code=1, priv_code=2,
                flip_frac_underpriv=0.15, flip_frac_priv=0.15, seed=42):
    rng = np.random.default_rng(seed)
    df2 = df.copy()
    mask_u = (df2[sex_col] == underpriv_code) & (df2[target_col] == 0)
    idx_u = df2[mask_u].index
    flip_idx_u = rng.choice(idx_u, size=int(len(idx_u) * flip_frac_underpriv), replace=False)
    df2.loc[flip_idx_u, target_col] = 1
    mask_p = (df2[sex_col] == priv_code) & (df2[target_col] == 1)
    idx_p = df2[mask_p].index
    flip_idx_p = rng.choice(idx_p, size=int(len(idx_p) * flip_frac_priv), replace=False)
    df2.loc[flip_idx_p, target_col] = 0
    return df2

df_biased = inject_bias(df_clean, underpriv_code=1, priv_code=2)
df_biased['AGE_GROUP'] = pd.factorize(df_biased['AGE_GROUP'])[0]

def fairness_metrics(y_true, y_pred, sensitive, priv_val, underpriv_val):
    def rates(mask):
        yt = np.asarray(y_true)[mask]; yp = np.asarray(y_pred)[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        ppr = (yp == 1).mean()
        return tpr, fpr, ppr
    sensitive = np.asarray(sensitive)
    tpr_p, fpr_p, ppr_p = rates(sensitive == priv_val)
    tpr_u, fpr_u, ppr_u = rates(sensitive == underpriv_val)
    return {
        'statistical_parity_diff': ppr_p - ppr_u,
        'TPR_priv (women)': tpr_p, 'TPR_underpriv (men)': tpr_u, 'TPR_diff (EOd)': tpr_p - tpr_u,
        'FPR_priv (women)': fpr_p, 'FPR_underpriv (men)': fpr_u, 'FPR_diff (EOd)': fpr_p - fpr_u,
    }

def split(seed):
    """Mismo split que la corrida anterior."""
    return train_test_split(df_biased, test_size=0.2, random_state=seed,
                            stratify=df_biased[target_col])

import uuid
def save_csv_atomic(d, path, **kw):
    """Guarda en un temporal con nombre único y renombra: nunca queda un archivo a medio escribir
    y dos sesiones no pueden pisarse el mismo temporal."""
    tmp = f'{path}.{uuid.uuid4().hex}.tmp'
    d.to_csv(tmp, **kw)
    os.replace(tmp, path)

print("Setup OK", df_biased.shape)
print("Default rate por sexo (original):", df_clean.groupby('SEX')[target_col].mean().round(4).to_dict())
print("Default rate por sexo (con sesgo):", df_biased.groupby('SEX')[target_col].mean().round(4).to_dict())
print("Default rate por edad, original (core = 25-65, edge = resto):",
      df_clean.groupby('AGE_GROUP')[target_col].mean().round(4).to_dict(), "| tamaños:", df_clean['AGE_GROUP'].value_counts().to_dict())


In [ ]:
# ===== CELDA COPIA · Reusar de v4 lo que no cambia (datos sintéticos y la seed 1 completa) =====
import shutil, uuid
reuse = ([f'synth_seed{s}.csv' for s in SEEDS] + [f'test_ids_seed{s}.csv' for s in SEEDS] +
         ['best_params_seed1.json', 'cv_results_seed1.csv', 'results_seed1.csv', 'data_rates_seed1.csv'])
for fn in reuse:
    src_f, dst_f = f'{V4}/{fn}', f'{OUT}/{fn}'
    if os.path.exists(src_f) and not os.path.exists(dst_f):
        tmp = f'{dst_f}.{uuid.uuid4().hex}.tmp'      # nombre único: los 3 notebooks pueden copiar a la vez
        shutil.copyfile(src_f, tmp)
        os.replace(tmp, dst_f)
listos = sum(os.path.exists(f'{OUT}/{fn}') for fn in reuse)
print(f'Archivos de v4 disponibles en v5: {listos} de {len(reuse)}')
assert listos == len(reuse), 'Falta algún archivo en v4: avisá antes de seguir'


In [ ]:
# ===== CELDA B · CV dentro de cada seed + 3 condiciones (CPU alcanza) =====
# Antes: Celda 0. Puede correr en otra sesión mientras la etapa A sigue con el GAN:
# procesa cada seed apenas aparece su synth_seedN.csv en Drive y espera las que faltan.
# Si se corta, volvé a correrla: las seeds completas se saltean y el CV ya hecho se reusa.
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

# ---------------- configuración ----------------
SEEDS = list(range(1, 11))   # todas las seeds (para la consolidación final)
RUN_SEEDS = [8, 9, 10]          # seeds que procesa ESTE notebook
TUNE_EVERY_SEED = True       # CV dentro de cada seed, sobre su propio train (diseño sin fuga)
N_ITER = 12              # combinaciones probadas por modelo
CV_FOLDS = 3
RW_MODELS = ['LogReg', 'RandomForest', 'GradBoost', 'XGBoost', 'LightGBM', 'CatBoost']

def build_models(seed):
    """Los mismos 9 modelos de antes; los hiperparámetros los pisa el CV."""
    return {
        "LogReg":       make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=seed)),
        "SVM":          make_pipeline(StandardScaler(), LinearSVC(max_iter=5000, random_state=seed)),
        "kNN":          make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        "MLP":          make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                                                                     early_stopping=True, random_state=seed)),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1),
        "GradBoost":    GradientBoostingClassifier(random_state=seed),
        "XGBoost":      XGBClassifier(eval_metric="logloss", random_state=seed),
        "LightGBM":     LGBMClassifier(random_state=seed, verbose=-1),
        "CatBoost":     CatBoostClassifier(random_state=seed, verbose=0, allow_writing_files=False),
    }

# espacios de búsqueda (chicos a propósito, para que entre en el día)
SPACES = {
    "LogReg":       {"logisticregression__C": loguniform(1e-3, 1e2)},
    "SVM":          {"linearsvc__C": loguniform(1e-3, 1e1)},
    "kNN":          {"kneighborsclassifier__n_neighbors": randint(5, 101),
                     "kneighborsclassifier__weights": ["uniform", "distance"]},
    "MLP":          {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],
                     "mlpclassifier__alpha": loguniform(1e-5, 1e-1),
                     "mlpclassifier__learning_rate_init": loguniform(3e-4, 3e-3)},
    "RandomForest": {"max_depth": [None, 6, 10, 16],
                     "min_samples_leaf": randint(1, 30),
                     "max_features": ["sqrt", 0.3, 0.5]},
    "GradBoost":    {"n_estimators": randint(100, 301), "learning_rate": loguniform(0.02, 0.3),
                     "max_depth": randint(2, 6), "subsample": uniform(0.7, 0.3)},
    "XGBoost":      {"n_estimators": randint(100, 501), "learning_rate": loguniform(0.02, 0.3),
                     "max_depth": randint(2, 8), "subsample": uniform(0.7, 0.3),
                     "colsample_bytree": uniform(0.6, 0.4), "min_child_weight": randint(1, 11)},
    "LightGBM":     {"n_estimators": randint(100, 501), "learning_rate": loguniform(0.02, 0.3),
                     "num_leaves": randint(8, 64), "min_child_samples": randint(10, 100),
                     "subsample": uniform(0.7, 0.3), "subsample_freq": [1],
                     "colsample_bytree": uniform(0.6, 0.4)},
    "CatBoost":     {"iterations": randint(200, 601), "learning_rate": loguniform(0.02, 0.3),
                     "depth": randint(4, 9), "l2_leaf_reg": loguniform(1, 10)},
}
EXPECTED_ROWS = 2 * len(SPACES) + len(RW_MODELS)   # 9 original + 9 synthetic + 6 reweighed = 24

# ---------------- funciones ----------------
def to_jsonable(params):
    out = {}
    for k, v in params.items():
        if hasattr(v, 'item'):
            v = v.item()
        if isinstance(v, tuple):
            v = list(v)
        out[k] = v
    return out

def from_json(params):
    return {k: (tuple(v) if isinstance(v, list) else v) for k, v in params.items()}

def tune(name, X, y, seed):
    search = RandomizedSearchCV(build_models(seed)[name], SPACES[name], n_iter=N_ITER,
                                scoring='roc_auc',
                                cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=seed),
                                random_state=seed, n_jobs=-1, refit=False, return_train_score=True)
    search.fit(X, y)
    return search.best_params_, search.best_score_, pd.DataFrame(search.cv_results_)

def compute_weights(X_tr, y_tr, sex_tr):
    tmp = X_tr.copy()
    tmp['label'] = y_tr.values
    tmp['SEX'] = sex_tr.values
    bld = BinaryLabelDataset(df=tmp, label_names=['label'], protected_attribute_names=['SEX'],
                             favorable_label=0, unfavorable_label=1)
    rw = Reweighing(unprivileged_groups=[{'SEX': 1}], privileged_groups=[{'SEX': 2}])
    return rw.fit_transform(bld).instance_weights

def score_of(model, X):
    """Score continuo para AUC: probabilidad si existe, si no decision_function (LinearSVC)."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def fit_eval(name, params, seed, X_tr, y_tr, X_te, y_te, sex_te, weights=None):
    model = build_models(seed)[name].set_params(**params)
    if weights is None:
        model.fit(X_tr, y_tr)
    elif hasattr(model, 'steps'):
        model.fit(X_tr, y_tr, **{model.steps[-1][0] + '__sample_weight': weights})
    else:
        model.fit(X_tr, y_tr, sample_weight=weights)
    y_pred = model.predict(X_te)
    m = fairness_metrics(y_te, y_pred, sex_te, priv_val=2, underpriv_val=1)
    m['EOd_max'] = max(abs(m['TPR_diff (EOd)']), abs(m['FPR_diff (EOd)']))
    m['accuracy'] = accuracy_score(y_te, y_pred)
    m['balanced_accuracy'] = balanced_accuracy_score(y_te, y_pred)
    m['roc_auc'] = roc_auc_score(y_te, score_of(model, X_te))
    m['pred_positive_rate'] = float(np.mean(y_pred))   # detecta modelos que predicen casi todo una clase
    return m, model, y_pred

def get_params(seed, X_tr, y_tr):
    """Reusa el CV si ya existe (cortes) o el de la seed 1 si TUNE_EVERY_SEED=False; si no, lo corre."""
    own = f'{OUT}/best_params_seed{seed}.json'
    shared = f'{OUT}/best_params_seed{SEEDS[0]}.json'
    for p in ([own] if TUNE_EVERY_SEED else [own, shared]):
        if os.path.exists(p):
            with open(p) as fh:
                print(f'  hiperparámetros reusados de {os.path.basename(p)}')
                return {k: from_json(v) for k, v in json.load(fh).items()}
    best, cv_all, t_cv = {}, [], time.time()
    for name in SPACES:
        t1 = time.time()
        bp, bs, cvr = tune(name, X_tr, y_tr, seed)
        best[name] = bp
        cvr['model'] = name
        cvr['seed'] = seed
        cv_all.append(cvr)
        print(f'  CV {name:12s} AUC={bs:.4f}  ({time.time() - t1:.0f}s)  {to_jsonable(bp)}')
    print(f'  CV total: {(time.time() - t_cv) / 60:.1f} min')
    save_csv_atomic(pd.concat(cv_all, ignore_index=True), f'{OUT}/cv_results_seed{seed}.csv', index=False)
    tmp = f'{own}.{uuid.uuid4().hex}.tmp'
    with open(tmp, 'w') as fh:
        json.dump({k: to_jsonable(v) for k, v in best.items()}, fh, indent=1)
    os.replace(tmp, own)
    return best

def group_rates(y, sex, w=None):
    """Tasa de default por sexo (ponderada si hay pesos) y brecha mujeres − hombres."""
    w = np.ones(len(y)) if w is None else np.asarray(w)
    y, sex = np.asarray(y), np.asarray(sex)
    r = {g: float(np.sum(w[sex == g] * y[sex == g]) / np.sum(w[sex == g])) for g in (1, 2)}
    return {'default_rate_men': r[1], 'default_rate_women': r[2], 'gap_women_minus_men': r[2] - r[1]}

def seed_done(seed):
    p = f'{OUT}/results_seed{seed}.csv'
    if not os.path.exists(p):
        return False
    r = pd.read_csv(p)
    return len(r) == EXPECTED_ROWS and {'roc_auc', 'balanced_accuracy'} <= set(r.columns)

def run_seed(seed):
    t0 = time.time()
    df_tr, df_te = split(seed)
    ids_path = f'{OUT}/test_ids_seed{seed}.csv'
    if os.path.exists(ids_path):
        assert (pd.read_csv(ids_path)['ID'].values == df_te['ID'].values).all(), \
            f'seed {seed}: el split no coincide con el de la etapa A'

    X_tr, y_tr, sex_tr = df_tr.drop(columns=drop_cols), df_tr[target_col], df_tr['SEX']
    X_te, y_te, sex_te = df_te.drop(columns=drop_cols), df_te[target_col], df_te['SEX']
    synth = pd.read_csv(f'{OUT}/synth_seed{seed}.csv')
    X_s = synth[X_tr.columns]                 # mismas columnas y orden que los datos reales
    y_s = synth['default'].astype(int)
    weights = compute_weights(X_tr, y_tr, sex_tr)

    # datos de entrenamiento de cada condición: tasa de default por sexo (ejemplo para el texto
    # y control de que FairFinGAN no invierta la desigualdad)
    rates = pd.DataFrame([
        {'seed': seed, 'data': 'original_train', **group_rates(y_tr, sex_tr)},
        {'seed': seed, 'data': 'reweighed_train', **group_rates(y_tr, sex_tr, weights)},
        {'seed': seed, 'data': 'synthetic_train', **group_rates(y_s, synth['SEX'])},
    ])
    save_csv_atomic(rates, f'{OUT}/data_rates_seed{seed}.csv', index=False)
    print(rates[['data', 'default_rate_men', 'default_rate_women', 'gap_women_minus_men']].round(4).to_string(index=False))

    params = get_params(seed, X_tr, y_tr)

    rows = []
    for name in SPACES:
        p = params[name]
        m, _, _ = fit_eval(name, p, seed, X_tr, y_tr, X_te, y_te, sex_te)
        rows.append({'seed': seed, 'dataset': 'original', 'model': name, **m})
        m, _, _ = fit_eval(name, p, seed, X_s, y_s, X_te, y_te, sex_te)
        rows.append({'seed': seed, 'dataset': 'synthetic', 'model': name, **m})
        if name in RW_MODELS:
            m, _, _ = fit_eval(name, p, seed, X_tr, y_tr, X_te, y_te, sex_te, weights=weights)
            rows.append({'seed': seed, 'dataset': 'reweighed', 'model': name, **m})

    res = pd.DataFrame(rows)
    assert len(res) == EXPECTED_ROWS
    save_csv_atomic(res, f'{OUT}/results_seed{seed}.csv', index=False)
    print(f'===== seed {seed} lista en {(time.time() - t0) / 60:.1f} min =====')

# ---------------- loop: procesa las seeds a medida que la etapa A las termina ----------------
pending = [s for s in RUN_SEEDS if not seed_done(s)]
while pending:
    ready = [s for s in pending if os.path.exists(f'{OUT}/synth_seed{s}.csv')]
    if not ready:
        print(time.strftime('%H:%M'), '· esperando la etapa A para las seeds', pending)
        time.sleep(120)
        continue
    seed = ready[0]
    print(f'===== seed {seed} =====')
    run_seed(seed)
    pending.remove(seed)
print("Etapa B terminada para las seeds", RUN_SEEDS)
